### Import thư viện

In [1]:
import random

### Trạng thái mục tiêu

In [2]:
goal_state = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

### Random trạng thái ban đầu

In [3]:
def input_state():
    numbers = list(range(9))  
    random.shuffle(numbers)

    state = [
        numbers[0:3],
        numbers[3:6],
        numbers[6:9]
    ]

    return state

### Các hàm xử lý trạng thái

In [4]:
def copy_state(state):
    return [row[:] for row in state]    

In [5]:
def print_state(state):
    for row in state:
        print(row)

In [6]:
def state_key(state):
    return tuple(tuple(row) for row in state)

In [7]:
def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return None, None

In [8]:
def is_goal_state(state):
    return state_key(state) == state_key(goal_state)

### Model

In [9]:
def get_possible_moves(state):
    zero_x, zero_y = find_zero(state)

    possible_moves = []

    if zero_x > 0:
        possible_moves.append("UP")

    if zero_x < 2:
        possible_moves.append("DOWN")

    if zero_y > 0:
        possible_moves.append("LEFT")

    if zero_y < 2:
        possible_moves.append("RIGHT")

    return possible_moves

In [10]:
def model(state, action):
    """
    model: mô hình dự đoán trạng thái mới
    input: trạng thái hiện tại + hành động
    output: trạng thái mới sau khi hành động
    """

    if action is None:
        return state

    new_state = copy_state(state)
    zero_x, zero_y = find_zero(new_state)

    new_x, new_y = zero_x, zero_y

    if action == "UP":
        new_x = zero_x - 1
    elif action == "DOWN":
        new_x = zero_x + 1
    elif action == "LEFT":
        new_y = zero_y - 1
    elif action == "RIGHT":
        new_y = zero_y + 1

    new_state[zero_x][zero_y], new_state[new_x][new_y] = (
        new_state[new_x][new_y],
        new_state[zero_x][zero_y]
    )

    return new_state

## Model-Based Reflex Agent

### Update-state

In [11]:
def update_state(previous_state, previous_action, percept, model):
    return percept

### Rule-match

In [12]:
def rule_match(state, rules, visited):
    """
    So khớp trạng thái hiện tại với tập luật.
    Luật:
    1. Nếu còn hướng đi đến trạng thái chưa từng đi -> chọn hướng đó.
    2. Nếu một số hướng bị lặp -> cảnh báo và tránh.
    3. Nếu tất cả hướng đều lặp -> dừng.
    """

    possible_moves = get_possible_moves(state)

    print("\nCác bước có thể đi:", possible_moves)

    if not possible_moves:
        return {
            "condition": "Không còn bước đi hợp lệ",
            "action": None
        }

    unvisited_moves = []
    duplicated_moves = []

    for action in possible_moves:
        predicted_state = model(state, action)
        key = state_key(predicted_state)

        if key in visited:
            duplicated_moves.append((action, visited[key]))
        else:
            unvisited_moves.append((action, predicted_state))

    if duplicated_moves:
        print("\nCẢNH BÁO!")
        print("Một số hướng sẽ quay lại trạng thái đã đi:")

        for action, duplicated_step in duplicated_moves:
            if duplicated_step == 0:
                print(f"- Hướng {action} bị trùng với TRẠNG THÁI BAN ĐẦU")
            else:
                print(f"- Hướng {action} bị trùng với STEP {duplicated_step}")

    if len(unvisited_moves) > 0:
        selected_action, predicted_state = random.choice(unvisited_moves)

        return {
            "condition": "Còn hướng chưa đi",
            "action": selected_action
        }

    return {
        "condition": "Tất cả hướng đều đã đi",
        "action": None
    }

### Model-based-reflex-agent

In [13]:
class ModelBasedReflexAgent:
    def __init__(self):
        self.state = None              # trạng thái hiện tại agent nhớ
        self.model = model             # mô hình dự đoán hành động
        self.rules = [
            "Còn hướng chưa đi",
            "Tất cả hướng đều đã đi",
            "Không còn bước đi hợp lệ"
        ]
        self.action = None             # hành động trước đó
        self.visited = {}              # bộ nhớ các trạng thái đã đi
        self.current_step = 0

    def program(self, percept):
        """
        MODEL-BASED-REFLEX-AGENT(percept) returns action
        """

        # state ← UPDATE-STATE(state, action, percept, model)
        self.state = update_state(
            self.state,
            self.action,
            percept,
            self.model
        )

        # lưu trạng thái hiện tại vào visited
        key = state_key(self.state)

        if key not in self.visited:
            self.visited[key] = self.current_step

        # rule ← RULE-MATCH(state, rules)
        rule = rule_match(self.state, self.rules, self.visited)

        print("\nRule được chọn:", rule["condition"])

        # action ← rule.ACTION
        self.action = rule["action"]

        # return action
        return self.action

### Main program

In [17]:
agent = ModelBasedReflexAgent()

current_state = input_state()

print("\nTrạng thái ban đầu:")
print_state(current_state)

print("\nTrạng thái đích cần đạt:")
print_state(goal_state)

step = 0

# Kiểm tra nếu trạng thái ban đầu đã là trạng thái đích
if is_goal_state(current_state):
    print("\nDỪNG LẠI!")
    print("Lý do: Trạng thái ban đầu đã là trạng thái đích.")
else:
    while True:
        print(f"\n========== LẦN ROLL {step + 1} ==========")

        agent.current_step = step

        percept = current_state

        action = agent.program(percept)

        if action is None:
            print("\nDỪNG LẠI!")
            print("Lý do: Không còn hướng nào dẫn tới trạng thái chưa đi.")
            break

        print("\nAgent chọn hành động:", action)

        current_state = model(current_state, action)

        step += 1

        print("\nTrạng thái sau khi thực hiện hành động:")
        print_state(current_state)

        if is_goal_state(current_state):
            print("\nDỪNG LẠI!")
            print("Lý do: Agent đã giải quyết xong bài toán.")
            print("Trạng thái hiện tại đã trùng với trạng thái đích.")
            break

print("\nTổng số lần roll:", step)


Trạng thái ban đầu:
[3, 7, 2]
[1, 5, 0]
[6, 4, 8]

Trạng thái đích cần đạt:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

========== LẦN ROLL 1 ==========

Các bước có thể đi: ['UP', 'DOWN', 'LEFT']

Rule được chọn: Còn hướng chưa đi

Agent chọn hành động: UP

Trạng thái sau khi thực hiện hành động:
[3, 7, 0]
[1, 5, 2]
[6, 4, 8]

========== LẦN ROLL 2 ==========

Các bước có thể đi: ['DOWN', 'LEFT']

CẢNH BÁO!
Một số hướng sẽ quay lại trạng thái đã đi:
- Hướng DOWN bị trùng với TRẠNG THÁI BAN ĐẦU

Rule được chọn: Còn hướng chưa đi

Agent chọn hành động: LEFT

Trạng thái sau khi thực hiện hành động:
[3, 0, 7]
[1, 5, 2]
[6, 4, 8]

========== LẦN ROLL 3 ==========

Các bước có thể đi: ['DOWN', 'LEFT', 'RIGHT']

CẢNH BÁO!
Một số hướng sẽ quay lại trạng thái đã đi:
- Hướng RIGHT bị trùng với STEP 1

Rule được chọn: Còn hướng chưa đi

Agent chọn hành động: DOWN

Trạng thái sau khi thực hiện hành động:
[3, 5, 7]
[1, 0, 2]
[6, 4, 8]

========== LẦN ROLL 4 ==========

Các bước có thể đi: ['UP', 'DOWN', 'LEFT